In [ ]:
import re
import time
from pathlib import Path
from datetime import datetime, timedelta, UTC
from typing import Union, List, Iterable
import logging

import pandas as pd
from pyspark.sql import functions as F

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import Configuration
from teehr.fetching.utils import (
    format_nwm_configuration_metadata,
    build_remote_nwm_filelist,
)
from teehr.fetching.const import (
    NWM30_ANALYSIS_CONFIG,
    NWM_VARIABLE_MAPPER,
    VARIABLE_NAME
)
from teehr.utils.utils import remove_dir_if_exists

from forcing_utils import cache_weights_view, parse_value_and_reference_times_from_path


logger = logging.getLogger()

In [ ]:
# Maps NWM version string to the corresponding analysis config dict
NWM_VERSION_ANALYSIS_CONFIG = {
    "nwm30": NWM30_ANALYSIS_CONFIG,
}
NWM30_WEIGHTS_VARIABLE_NAME = "rainfall_hourly_rate"

In [ ]:
shuffle_partitions = 200

start_dt = datetime(year=2023, month=9, day=19, hour=0) # "2023-09-19 00:00"
end_dt = datetime(year=2026, month=4, day=29, hour=23)   # "2026-04-29 23:00"

nwm_version = "nwm30"
nwm_variable_name = "RAINRATE"

teehr_unit_name = "mm/s"
location_id_prefix = "usgsbasin"

file_chunk_size = 100

#---------------------------------------------------------------#
target_table_name = "primary_timeseries"
# nwm_configuration = "forcing_analysis_assim"
# nwm_configuration = "forcing_analysis_assim_extend"

# nwm_configuration = "forcing_analysis_assim_alaska"
# nwm_configuration = "forcing_analysis_assim_extend_alaska"  # <--- MISSING?
# nwm_configuration = "forcing_analysis_assim_hawaii"
nwm_configuration = "forcing_analysis_assim_puertorico"

#---------------------------------------------------------------#
# target_table_name = "secondary_timeseries"
# nwm_configuration = "forcing_short_range"
# nwm_configuration = "forcing_short_range_alaska"
# nwm_configuration = "forcing_short_range_hawaii"
# nwm_configuration = "forcing_short_range_puertorico"
# nwm_configuration = "forcing_medium_range"
# nwm_configuration = "forcing_medium_range_alaska"
# nwm_configuration = "forcing_medium_range_blend"
# nwm_configuration = "forcing_medium_range_blend_alaska"

In [ ]:
ev_config = format_nwm_configuration_metadata(
    nwm_config_name=nwm_configuration,
    nwm_version=nwm_version,
)

teehr_config_name = ev_config["name"]
variable_mapper = NWM_VARIABLE_MAPPER
teehr_variable_name = variable_mapper[VARIABLE_NAME].get(
        nwm_variable_name, {}
).get("name", nwm_variable_name)

analysis_config_dict = NWM_VERSION_ANALYSIS_CONFIG.get(nwm_version, NWM30_ANALYSIS_CONFIG)
is_analysis = nwm_configuration in analysis_config_dict

if target_table_name == "primary_timeseries":
    output_type = "primary"
    latest_time_field_name = "value_time"
else:
    output_type = "secondary"
    latest_time_field_name = "reference_time"

In [ ]:
%%time
dir_path =  "/data/data_processing/backfill-nwm/nwm-forcing-teehr-warehouse"

spark = create_spark_session(
    start_spark_cluster=False,
    executor_instances=8,
    executor_cores=7,
    executor_memory="50g",
    enable_gcs=True,
    gcs_project_id="anonymous",
    update_configs = {
        "spark.sql.shuffle.partitions": shuffle_partitions
    },    
)
ev = teehr.LocalReadWriteEvaluation(
    spark=spark,
    dir_path=dir_path,
    create_dir=True
)

In [ ]:
nwm_cache_dir = Path(ev.cache_dir) / "fetching" / ev_config["name"]
remove_dir_if_exists(nwm_cache_dir)
nwm_cache_dir.mkdir(parents=True, exist_ok=True)

if "hawaii" in nwm_configuration:
    weights_domain_name = "nwm30_hawaii_forcing"
elif "alaska" in nwm_configuration:
    weights_domain_name = "nwm30_alaska_forcing"
elif "puertorico" in nwm_configuration:
    weights_domain_name = "nwm30_puertorico_forcing"
else:
    weights_domain_name = "nwm30_conus_forcing"

In [ ]:
%%time
cache_weights_view(
    ev=ev,
    location_id_prefix=location_id_prefix,
    weights_domain_name=weights_domain_name,
    weights_variable_name=NWM30_WEIGHTS_VARIABLE_NAME,
)

In [ ]:
# Get the min/max position_index from the weights use to filter the RAINRATE grid.
# # In cache_weights_view, after CACHE TABLE.
# pos_range = ev.spark.sql(
#     "SELECT MIN(position_index) as min_pos, MAX(position_index) as max_pos FROM fractions_view"
# ).collect()[0]

In [ ]:
%%time
filepaths = build_remote_nwm_filelist(
    configuration=nwm_configuration,
    output_type="forcing",
    start_dt=start_dt,
    end_dt=end_dt,
    analysis_config_dict=analysis_config_dict,
    t_minus_hours=None,
    ignore_missing_file=False,
    prioritize_analysis_value_time=False,
    drop_overlapping_assimilation_values=True,
    ingest_days=None
)
# For sedona the filepath must start with 'gs' instead of 'gcs'
filepaths = sorted([fp.replace("gcs://", "gs://") for fp in filepaths])
chunked_filepaths = [
    filepaths[i:i+file_chunk_size] for i in range(0, len(filepaths), file_chunk_size)
]

In [ ]:
%%time
logger.info(
    f"Processing {len(filepaths)} file(s) in {len(chunked_filepaths)} chunk(s) "
    f"of up to {file_chunk_size} file(s) per chunk"
)
for filepath_chunk in chunked_filepaths:
    logger.info(
        f"Chunk {filepath_chunk[0]} – {filepath_chunk[-1]}: "
        f"{len(filepath_chunk)} file(s)"
    )
    chunk_start = f"{Path(filepath_chunk[0]).parent.parent.name}_{Path(filepath_chunk[0]).stem}"
    chunk_end = f"{Path(filepath_chunk[-1]).parent.parent.name}_{Path(filepath_chunk[-1]).stem}"

    t0 = time.time()
    logger.info(f"Processing {len(filepath_chunk)} NWM forcing files")
    # Ensure filepaths are list of strings
    fpath_list = [str(fp) for fp in filepath_chunk]
    # Read NetCDF files via Sedona and extract the target variable as a raster
    nc_sdf = (
        ev.spark.read.format("binaryFile")
        .load(fpath_list)
        .selectExpr(
            f"RS_FromNetCDF(content, '{nwm_variable_name}', 'x', 'y') as raster",
            "path as filepath",
        )
    )
    logger.info("Parsing value_time and reference_time from filepaths.")
    nc_sdf = parse_value_and_reference_times_from_path(nc_sdf, is_analysis)
    
    logger.info("Exploding raster pixels.")
    raster_exp_sdf = nc_sdf.selectExpr(
        "posexplode(RS_BandAsArray(raster, 1))",
        "value_time",
        "reference_time",
    ).selectExpr(
        "value_time",
        "reference_time",
        "col as value",
        "CAST(pos AS BIGINT) as position_index",
    )
    raster_exp_sdf.createOrReplaceTempView("raster_values")

    # Compute coverage-weighted average per location and timestep
    map_results = ev.spark.sql(f"""
        SELECT /*+ BROADCAST(w) */
            w.location_id,
            CAST(r.value_time AS TIMESTAMP_NTZ) AS value_time,
            CAST(SUM(r.value * w.fraction_covered) / SUM(w.fraction_covered) AS FLOAT) AS value,
            CAST(r.reference_time AS TIMESTAMP_NTZ) AS reference_time,
            '{teehr_unit_name}' AS unit_name,
            '{teehr_variable_name}' AS variable_name,
            '{teehr_config_name}' AS configuration_name
        FROM
            raster_values AS r
        JOIN
            fractions_view AS w ON r.position_index = w.position_index
        GROUP BY
            w.location_id, r.value_time, r.reference_time
    """)    

    if output_type == "secondary":
        map_results = map_results.withColumn("member", F.lit(member).cast("string"))
    out_path = nwm_cache_dir / f"{chunk_start}_to_{chunk_end}"
    map_results.write.parquet(out_path.as_posix())
    logger.info(f"Wrote chunk to cache: {out_path.name}")

    ev.spark.catalog.dropTempView("raster_values")
    elapsed = (time.time() - t0) / 60
    logger.info(
        f"--> Processed {len(filepath_chunk)} files and wrote MAP to cache in {elapsed:.2f} min"
    )

In [ ]:
raster_exp_sdf.columns

In [ ]:
raster_exp_sdf.select(F.min("position_index"), F.max("position_index")).show()